# 01 - Ingestão Bronze

## Contexto

**Objetivo desta etapa (camada Bronze da Arquitetura Medalhão):** trazer o arquivo bruto da fonte de dados para dentro do Lakehouse como tabela Delta, **sem nenhuma limpeza ou transformação** — só o dado como ele veio, mais metadados de controle (data de ingestão e arquivo de origem).

## Fonte de dados

| | |
|---|---|
| Arquivo | `games_march2025_full.csv` |
| Origem | [Steam Games Dataset 2025](https://www.kaggle.com/datasets/artermiloff/steam-games-dataset) (Artemiy Ermilov, Kaggle) |
| Licença | MIT |
| Conteúdo | Coleta bruta de 94.948 jogos da Steam (scraping + Steam API/SteamSpy), com preço, plataformas suportadas, gêneros/tags, faixa etária, DLCs, nota da crítica e popularidade estimada |
| Chave do jogo | `appid` |

Todas as colunas são lidas como `string` de propósito — a tipagem correta (número, data, booleano) é responsabilidade da camada **Silver**, não da Bronze.

## Setup: catálogos das três camadas

In [0]:
# Cria os três catálogos (um por camada da Arquitetura Medalhão) e, dentro de cada um, o schema steam_games que
# agrupa as tabelas do projeto.
spark.sql("CREATE CATALOG IF NOT EXISTS bronze")
spark.sql("CREATE CATALOG IF NOT EXISTS silver")
spark.sql("CREATE CATALOG IF NOT EXISTS gold")

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze.steam_games")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver.steam_games")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold.steam_games")

DataFrame[]

## Parâmetros

In [0]:
# Define o caminho do CSV no Volume e o nome da tabela Bronze de destino.
from pyspark.sql import functions as F

VOLUME_PATH = "/Volumes/bronze/steam_games/raw_files"
games_full_csv_path = f"{VOLUME_PATH}/games_march2025_full.csv"
games_full_raw_table = "bronze.steam_games.games_full_raw"

## Ingestão

Arquivo grande (~471 MB) e "raw": usamos `multiLine` e `escape` para lidar com campos de texto longos (descrições) que podem ter aspas e quebras de linha internas.

In [0]:
# Lê o CSV bruto do Volume com todas as colunas como string (sem inferir tipos) e acrescenta duas colunas de
# controle: data/hora da ingestão e arquivo de origem. Depois mostra a contagem de linhas, o schema e uma
# amostra de 2 linhas.
df_games_full = (
    spark.read
    .option("header", "true")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .csv(games_full_csv_path)
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_source_file", F.lit("games_march2025_full.csv"))
)

print("Linhas lidas:", df_games_full.count())
df_games_full.printSchema()
display(df_games_full.limit(2))

Linhas lidas: 94948
root
 |-- appid: string (nullable = true)
 |-- name: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- required_age: string (nullable = true)
 |-- price: string (nullable = true)
 |-- dlc_count: string (nullable = true)
 |-- detailed_description: string (nullable = true)
 |-- about_the_game: string (nullable = true)
 |-- short_description: string (nullable = true)
 |-- reviews: string (nullable = true)
 |-- header_image: string (nullable = true)
 |-- website: string (nullable = true)
 |-- support_url: string (nullable = true)
 |-- support_email: string (nullable = true)
 |-- windows: string (nullable = true)
 |-- mac: string (nullable = true)
 |-- linux: string (nullable = true)
 |-- metacritic_score: string (nullable = true)
 |-- metacritic_url: string (nullable = true)
 |-- achievements: string (nullable = true)
 |-- recommendations: string (nullable = true)
 |-- notes: string (nullable = true)
 |-- supported_languages: string (nullable = t

appid,name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews,header_image,website,support_url,support_email,windows,mac,linux,metacritic_score,metacritic_url,achievements,recommendations,notes,supported_languages,full_audio_languages,packages,developers,publishers,categories,genres,screenshots,movies,user_score,score_rank,positive,negative,estimated_owners,average_playtime_forever,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,discount,peak_ccu,tags,pct_pos_total,num_reviews_total,pct_pos_recent,num_reviews_recent,_ingestion_timestamp,_source_file
730,Counter-Strike 2,2012-08-21,0,0.0,1,"For over two decades, Counter-Strike has offered an elite competitive experience, one shaped by millions of players from across the globe. And now the next chapter in the CS story is about to begin. This is Counter-Strike 2. A free upgrade to CS:GO, Counter-Strike 2 marks the largest technical leap in Counter-Strike’s history. Built on the Source 2 engine, Counter-Strike 2 is modernized with realistic physically-based rendering, state of the art networking, and upgraded Community Workshop tools. In addition to the classic objective-focused gameplay that Counter-Strike pioneered in 1999, Counter-Strike 2 features: All-new CS Ratings with the updated Premier mode Global and Regional leaderboards Upgraded and overhauled maps Game-changing dynamic smoke grenades Tick-rate-independent gameplay Redesigned visual effects and audio All items from CS:GO moving forward to CS2","For over two decades, Counter-Strike has offered an elite competitive experience, one shaped by millions of players from across the globe. And now the next chapter in the CS story is about to begin. This is Counter-Strike 2. A free upgrade to CS:GO, Counter-Strike 2 marks the largest technical leap in Counter-Strike’s history. Built on the Source 2 engine, Counter-Strike 2 is modernized with realistic physically-based rendering, state of the art networking, and upgraded Community Workshop tools. In addition to the classic objective-focused gameplay that Counter-Strike pioneered in 1999, Counter-Strike 2 features: All-new CS Ratings with the updated Premier mode Global and Regional leaderboards Upgraded and overhauled maps Game-changing dynamic smoke grenades Tick-rate-independent gameplay Redesigned visual effects and audio All items from CS:GO moving forward to CS2","For over two decades, Counter-Strike has offered an elite competitive experience, one shaped by millions of players from across the globe. And now the next chapter in the CS story is about to begin. This is Counter-Strike 2.",null,https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/730/header.jpg?t=1729703045,http://counter-strike.net/,null,null,True,False,True,0,null,1,4401572,Includes intense violence and blood.,"['Czech', 'Danish', 'Dutch', 'English', 'Finnish', 'French', 'German', 'Hungarian', 'Italian', 'Japanese', 'Korean', 'Norwegian', 'Polish', 'Portuguese - Portugal', 'Portuguese - Brazil', 'Romanian', 'Russian', 'Simplified Chinese', 'Spanish - Spain', 'Swedish', 'Thai', 'Traditional Chinese', 'Turkish', 'Bulgarian', 'Ukrainian', 'Greek', 'Spanish - Latin America', 'Vietnamese', 'Indonesian']","['English', 'Indonesian']","[{'title': 'Buy Counter-Strike 2', 'description': '', 'subs': [{'text': 'Counter-Strike 2 - Free', 'description': '', 'price': 0.0}, {'text': 'Prime Status Upgrade - $14.99', 'description': '', 'price': 14.99}]}]",['Valve'],['Valve'],"['Multi-player', 'Cross-Platform Multiplayer', 'Steam Trading Cards', 'Steam Workshop', 'In-App Purchases', 'Valve Anti-Cheat enabled', 'Stats', 'Remote Play on Phone', 'Remote Play on Tablet', 'Remote Play on TV', 'Steam Timeline']","['Action', 'Free To Play']","['https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/730/ss_796601d9d67faf53486eeb26d0724347cea67ddc.1920x1080.jpg?t=1729703045', 'https://shared.akamai.steamstatic.com/store_item_assets/stea

In [0]:
# Grava o DataFrame como tabela Delta gerenciada (bronze.steam_games.games_full_raw), sobrescrevendo se já
# existir, e confirma quantas linhas foram gravadas.
(
    df_games_full.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(games_full_raw_table)
)

print(f"Tabela {games_full_raw_table} criada com {spark.table(games_full_raw_table).count()} linhas.")

Tabela bronze.steam_games.games_full_raw criada com 94948 linhas.


## Validação da ingestão

Contagem de linhas da tabela Bronze, comparada com a contagem original do arquivo-fonte (validada previamente fora do Databricks): **94.948** linhas. Número batendo confirma que a leitura do CSV (aspas, quebras de linha) não corrompeu nenhuma linha.

In [0]:
# Conta as linhas da tabela Bronze para comparar com o arquivo original (94.948).
display(spark.sql(f"SELECT COUNT(*) AS linhas FROM {games_full_raw_table}"))

linhas
94948
